# Databricks Concurrency Benchmark - Iceberg

This notebook demonstrates a comprehensive concurrency benchmarking workflow using Databricks SQL and Iceberg tables. The benchmark involves loading large datasets from external storage, transforming data using VARIANT columns, and creating structured Iceberg tables for performance testing.

## Overview
- **Dataset**: Line items data (~60M records)
- **Storage**: Azure Data Lake Storage Gen2
- **Format**: Parquet to Iceberg transformation
- **Focus**: Concurrency and performance benchmarking

## 1. Setup Environment and Configuration

First, we'll import the necessary libraries and configure our environment for connecting to Databricks.

## 2. Explore External Storage Locations

Let's start by examining the external storage locations and exploring the source data structure.

In [0]:
%sql
-- Show available external locations
SHOW EXTERNAL LOCATIONS;

name,url,comment
dbxdl-dataeng-storage-account-read-only,abfss://dataeng@dbxdl.dfs.core.windows.net/lineitems,null
dbxdl-delta-storage-account,abfss://delta@dbxdl.dfs.core.windows.net/,null
dbxdl-demo-storage-account-read-only,abfss://demo@dbxdl.dfs.core.windows.net/,null
dbxdl-warehouse-storage-account-read-only,abfss://warehouse@dbxdl.dfs.core.windows.net/,null
metastore_root_location,abfss://metastore@dbxmeta.dfs.core.windows.net/unitycatalog,"Auto-created external location which provides access to the nominated metastore-level storage account for the metastore. Changing the URL on this external location will not update the metastore-level storage, and could break access. You can update the credential on this external location if desired."


In [0]:
%sql
-- List files in the source location to understand data structure
LIST 'abfss://demo@dbxdl.dfs.core.windows.net/lineitems/' WITH (CREDENTIAL `dbxdl-storage-account-creds`);

path,name,size,modification_time
abfss://demo@dbxdl.dfs.core.windows.net/lineitems/data_0_0_0.snappy.parquet,data_0_0_0.snappy.parquet,118872603,1747552700000


## 3. Create and Configure Catalog Structure

Now we'll set up the catalog and schema structure for our benchmark test.

In [0]:
%sql
-- Show existing catalogs
SHOW CATALOGS;

catalog
02_de_dbt
02_lakeflow_dp
04_ai_semantics
04_concurrency
__databricks_internal
hive_metastore
lakebase-test
samples
system


In [0]:
%sql
-- Create the catalog for concurrency testing
DROP CATALOG IF EXISTS 05_CONCURRENCY_ICEBERG CASCADE;
CREATE CATALOG IF NOT EXISTS 05_CONCURRENCY_ICEBERG;

In [0]:
%sql
-- Set the catalog and schema context
USE CATALOG 05_CONCURRENCY_ICEBERG;
USE SCHEMA default;

### Optional: Custom Schema Location
If you need the schema in a different location than the default catalog location, you can uncomment and run the following command:

In [0]:
%sql
-- Uncomment if you need a custom schema location
-- CREATE SCHEMA IF NOT EXISTS BRONZE MANAGED LOCATION "abfss://delta@dbxdl.dfs.core.windows.net/default/";

## 4. Examine Source Parquet Files

Let's read a few sample records from the source Parquet files to understand the data structure.

In [0]:
%sql
-- Reading few records from the source Parquet files
SELECT * FROM PARQUET.`abfss://warehouse@dbxdl.dfs.core.windows.net/lineitems/*` LIMIT 5;

L_ORDERKEY,L_PARTKEY,L_SUPPKEY,L_LINENUMBER,L_QUANTITY,L_EXTENDEDPRICE,L_DISCOUNT,L_TAX,L_RETURNFLAG,L_LINESTATUS,L_SHIPDATE,L_COMMITDATE,L_RECEIPTDATE,L_SHIPINSTRUCT,L_SHIPMODE,L_COMMENT
52618212,1651955,1988,4,20.00,38137.40,0.09,0.01,A,F,1992-04-17,1992-04-25,1992-04-19,DELIVER IN PERSON,TRUCK,"e. ironic, expre"
52618212,397072,72082,5,28.00,32733.68,0.05,0.02,A,F,1992-03-09,1992-04-02,1992-03-30,DELIVER IN PERSON,SHIP,bove the daringly ironic dep
52618212,1586047,11063,6,10.00,11329.70,0.06,0.04,A,F,1992-04-16,1992-05-13,1992-05-15,NONE,MAIL,blithely reg
52618213,585172,10178,1,40.00,50286.00,0.05,0.06,A,F,1994-12-09,1994-10-15,1994-12-14,NONE,FOB,sauternes.
52618213,983216,8226,2,24.00,31180.08,0.05,0.07,A,F,1994-10-29,1994-10-25,1994-11-24,TAKE BACK RETURN,RAIL,ly ironic theodolites are


## 5. Create Raw Iceberg Table with VARIANT Column

We'll create a raw table using the VARIANT data type to store the semi-structured data from Parquet files.

In [0]:
%sql
-- Creating RAW_LINEITEMS_WAREHOUSE table to land the data
CREATE OR REPLACE TABLE 05_CONCURRENCY_ICEBERG.default.RAW_LINEITEMS_WAREHOUSE (
  V VARIANT
) USING ICEBERG;

In [0]:
%sql
-- Verify the table is empty initially
SELECT * FROM 05_CONCURRENCY_ICEBERG.default.RAW_LINEITEMS_WAREHOUSE LIMIT 10;

V


## 6. Load Data Using COPY INTO

⚠️ **Important Performance Note**: 
- COPY INTO must list the source path every run to decide what's new
- Listing millions of files on ADLS Gen2 becomes slow and expensive (many list calls)
- Consider Auto Loader's clean source options to DELETE/MOVE files after ~30 days

In [0]:
%sql
-- Copying the data from Parquet files into the Iceberg table
COPY INTO 05_CONCURRENCY_ICEBERG.default.RAW_LINEITEMS_WAREHOUSE
FROM (
  SELECT parse_json(to_json(struct(*))) AS V 
  FROM 'abfss://warehouse@dbxdl.dfs.core.windows.net/lineitems/*'
)
FILEFORMAT = PARQUET
FORMAT_OPTIONS ('singleVariantColumn' = 'true');

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
59986052,59986052,0


## 7. Validate Data Load and Count Records

Let's verify the data was loaded successfully and count the total records.

In [0]:
%sql
-- Count total records (should be around 59,986,052)
SELECT COUNT(*) FROM 05_CONCURRENCY_ICEBERG.default.RAW_LINEITEMS_WAREHOUSE;

COUNT(*)
59986052


In [0]:
%sql
-- View sample records to verify structure
SELECT * FROM 05_CONCURRENCY_ICEBERG.default.RAW_LINEITEMS_WAREHOUSE LIMIT 10;

V
"{""L_COMMENT"":""al excuses thrash furiously rea"",""L_COMMITDATE"":""1995-06-30"",""L_DISCOUNT"":0.04,""L_EXTENDEDPRICE"":35039.4,""L_LINENUMBER"":1,""L_LINESTATUS"":""O"",""L_ORDERKEY"":59184515,""L_PARTKEY"":1799069,""L_QUANTITY"":30,""L_RECEIPTDATE"":""1995-08-02"",""L_RETURNFLAG"":""N"",""L_SHIPDATE"":""1995-07-18"",""L_SHIPINSTRUCT"":""COLLECT COD"",""L_SHIPMODE"":""TRUCK"",""L_SUPPKEY"":74121,""L_TAX"":0.05}"
"{""L_COMMENT"":""y. quickly final accounts affix furiously i"",""L_COMMITDATE"":""1995-06-03"",""L_DISCOUNT"":0.05,""L_EXTENDEDPRICE"":64040.55,""L_LINENUMBER"":2,""L_LINESTATUS"":""O"",""L_ORDERKEY"":59184515,""L_PARTKEY"":950779,""L_QUANTITY"":35,""L_RECEIPTDATE"":""1995-07-11"",""L_RETURNFLAG"":""N"",""L_SHIPDATE"":""1995-07-08"",""L_SHIPINSTRUCT"":""COLLECT COD"",""L_SHIPMODE"":""SHIP"",""L_SUPPKEY"":75789,""L_TAX"":0.01}"
"{""L_COMMENT"":""uriously pending instructions."",""L_COMMITDATE"":""1995-06-21"",""L_DISCOUNT"":0.09,""L_EXTENDEDPRICE"":51315.88,""L_LINENUMBER"":3,""L_LINESTATUS"":""F"",""L_ORDERKEY"":59184515,""L_PARTKEY"":1736796,""L_QUANTITY"":28,""L_RECEIPTDATE"":""1995-05-09"",""L_RETURNFLAG"":""R"",""L_SHIPDATE"":""1995-05-07"",""L_SHIPINSTRUCT"":""NONE"",""L_SHIPMODE"":""AIR"",""L_SUPPKEY"":61814,""L_TAX"":0}"
"{""L_COMMENT"":""ions cajole carefully fluffily regul"",""L_COMMITDATE"":""1995-07-08"",""L_DISCOUNT"":0.1,""L_EXTENDEDPRICE"":71665.09,""L_LINENUMBER"":4,""L_LINESTATUS"":""F"",""L_ORDERKEY"":59184515,""L_PARTKEY"":324642,""L_QUANTITY"":43,""L_RECEIPTDATE"":""1995-06-23"",""L_RETURNFLAG"":""N"",""L_SHIPDATE"":""1995-06-07"",""L_SHIPINSTRUCT"":""DELIVER IN PERSON"",""L_SHIPMODE"":""SHIP"",""L_SUPPKEY"":24643,""L_TAX"":0.04}"
"{""L_COMMENT"":""s. quickly ironic dep"",""L_COMMITDATE"":""1995-06-19"",""L_DISCOUNT"":0.01,""L_EXTENDEDPRICE"":62013.42,""L_LINENUMBER"":5,""L_LINESTATUS"":""O"",""L_ORDERKEY"":59184515,""L_PARTKEY"":253523,""L_QUANTITY"":42,""L_RECEIPTDATE"":""1995-08-09"",""L_RETURNFLAG"":""N"",""L_SHIPDATE"":""1995-07-31"",""L_SHIPINSTRUCT"":""TAKE BACK RETURN"",""L_SHIPMODE"":""REG AIR"",""L_SUPPKEY"":78526,""L_TAX"":0.03}"
"{""L_COMMENT"":""ely express theo"",""L_COMMITDATE"":""1995-06-07"",""L_DISCOUNT"":0.07,""L_EXTENDEDPRICE"":16413.57,""L_LINENUMBER"":6,""L_LINESTATUS"":""F"",""L_ORDERKEY"":59184515,""L_PARTKEY"":950773,""L_QUANTITY"":9,""L_RECEIPTDATE"":""1995-06-14"",""L_RETURNFLAG"":""R"",""L_SHIPDATE"":""1995-06-02"",""L_SHIPINSTRUCT"":""DELIVER IN PERSON"",""L_SHIPMODE"":""TRUCK"",""L_SUPPKEY"":75783,""L_TAX"":0.07}"
"{""L_COMMENT"":""ly unusual requests. enticin"",""L_COMMITDATE"":""1996-11-11"",""L_DISCOUNT"":0.1,""L_EXTENDEDPRICE"":37215.62,""L_LINENUMBER"":1,""L_LINESTATUS"":""O"",""L_ORDERKEY"":59184516,""L_PARTKEY"":1109422,""L_QUANTITY"":26,""L_RECEIPTDATE"":""1996-09-26"",""L_RETURNFLAG"":""N"",""L_SHIPDATE"":""1996-09-20"",""L_SHIPINSTRUCT"":""COLLECT COD"",""L_SHIPMODE"":""MAIL"",""L_SUPPKEY"":34434,""L_TAX"":0.05}"
"{""L_COMMENT"":""e blithely. quickly final packages haggle "",""L_COMMITDATE"":""1996-10-15"",""L_DISCOUNT"":0.1,""L_EXTENDEDPRICE"":37594.22,""L_LINENUMBER"":2,""L_LINESTATUS"":""O"",""L_ORDERKEY"":59184516,""L_PARTKEY"":48068,""L_QUANTITY"":37,""L_RECEIPTDATE"":""1996-11-28"",""L_RETURNFLAG"":""N"",""L_SHIPDATE"":""1996-11-24"",""L_SHIPINSTRUCT"":""TAKE BACK RETURN"",""L_SHIPMODE"":""MAIL"",""L_SUPPKEY"":73069,""L_TAX"":0.03}"
"{""L_COMMENT"":""oss the slyly "",""L_COMMITDATE"":""1993-04-25"",""L_DISCOUNT"":0.07,""L_EXTENDEDPRICE"":45025.47,""L_LINENUMBER"":1,""L_LINESTATUS"":""F"",""L_ORDERKEY"":59184517,""L_PARTKEY"":915652,""L_QUANTITY"":27,""L_RECEIPTDATE"":""1993-03-05"",""L_RETURNFLAG"":""R"",""L_SHIPDATE"":""1993-03-04"",""L_SHIPINSTRUCT"":""DELIVER IN PERSON"",""L_SHIPMODE"":""REG AIR"",""L_SUPPKEY"":65671,""L_TAX"":0.03}"
"{""L_COMMENT"":""aggle fluffily carefull"",""L_COMMITDATE"":""1993-04-01"",""L_DISCOUNT"":0,""L_EXTENDEDPRICE"":14629.59,""L_LINENUMBER"":2,""L_LINESTATUS"":""F"",""L_ORDERKEY"":59184517,""L_PARTKEY"":1355570,""L_

## 8. Extract Fields from VARIANT Data

Let's extract and examine specific fields from the VARIANT column using the `variant_get()` function.

In [0]:
%sql
-- Select 5 random values to examine the data structure
SELECT
  variant_get(V, '$.L_ORDERKEY') AS order_key,
  variant_get(V, '$.L_PARTKEY') AS part_key,
  variant_get(V, '$.L_SUPPKEY') AS supp_key,
  variant_get(V, '$.L_LINENUMBER') AS line_number,
  variant_get(V, '$.L_QUANTITY') AS quantity,
  variant_get(V, '$.L_EXTENDEDPRICE') AS extended_price,
  variant_get(V, '$.L_DISCOUNT') AS discount,
  variant_get(V, '$.L_TAX') AS tax,
  variant_get(V, '$.L_RETURNFLAG') AS return_flag,
  variant_get(V, '$.L_LINESTATUS') AS line_status,
  variant_get(V, '$.L_SHIPDATE') AS ship_date,
  variant_get(V, '$.L_COMMITDATE') AS commit_date,
  variant_get(V, '$.L_RECEIPTDATE') AS receipt_date,
  variant_get(V, '$.L_SHIPINSTRUCT') AS ship_instruct,
  variant_get(V, '$.L_SHIPMODE') AS ship_mode,
  variant_get(V, '$.L_COMMENT') AS comment
FROM 05_CONCURRENCY_ICEBERG.default.RAW_LINEITEMS_WAREHOUSE
ORDER BY RAND() LIMIT 5;

order_key,part_key,supp_key,line_number,quantity,extended_price,discount,tax,return_flag,line_status,ship_date,commit_date,receipt_date,ship_instruct,ship_mode,comment
51300546,902448,52467,2,26,37710.4,0.01,0.07,"""R""","""F""","""1992-08-30""","""1992-08-16""","""1992-09-02""","""NONE""","""REG AIR""","""onic multip"""
34912865,405620,80633,4,45,68652,0.1,0.02,"""A""","""F""","""1995-02-18""","""1995-04-13""","""1995-03-16""","""DELIVER IN PERSON""","""AIR""",""" packages haggle carefully. slyly """
347296,887500,87501,2,29,43136.34,0.05,0.04,"""N""","""O""","""1998-04-27""","""1998-03-18""","""1998-05-24""","""DELIVER IN PERSON""","""REG AIR""","""lyly slyly final ideas. ironic attai"""
24008352,1395871,70911,2,19,37369.39,0.06,0.02,"""N""","""O""","""1997-06-20""","""1997-04-02""","""1997-07-19""","""NONE""","""AIR""","""ic platelets kindle express i"""
5850336,1455096,55097,3,48,50448.96,0.07,0.01,"""N""","""O""","""1996-12-24""","""1997-02-02""","""1997-01-05""","""TAKE BACK RETURN""","""AIR""","""nts x-ray care"""


In [0]:
%sql
-- Query records with specific ORDERKEY values (requires casting)
SELECT
  variant_get(V, '$.L_ORDERKEY') AS order_key,
  variant_get(V, '$.L_PARTKEY') AS part_key,
  variant_get(V, '$.L_SUPPKEY') AS supp_key,
  variant_get(V, '$.L_LINENUMBER') AS line_number,
  variant_get(V, '$.L_QUANTITY') AS quantity,
  variant_get(V, '$.L_EXTENDEDPRICE') AS extended_price,
  variant_get(V, '$.L_DISCOUNT') AS discount,
  variant_get(V, '$.L_TAX') AS tax,
  variant_get(V, '$.L_RETURNFLAG') AS return_flag,
  variant_get(V, '$.L_LINESTATUS') AS line_status,
  variant_get(V, '$.L_SHIPDATE') AS ship_date,
  variant_get(V, '$.L_COMMITDATE') AS commit_date,
  variant_get(V, '$.L_RECEIPTDATE') AS receipt_date,
  variant_get(V, '$.L_SHIPINSTRUCT') AS ship_instruct,
  variant_get(V, '$.L_SHIPMODE') AS ship_mode,
  variant_get(V, '$.L_COMMENT') AS comment
FROM 05_CONCURRENCY_ICEBERG.default.RAW_LINEITEMS_WAREHOUSE
WHERE order_key IN (
--WHERE CAST(variant_get(V, '$.L_ORDERKEY') AS STRING) IN (
  '30370724',
  '43675749',
  '46386755',
  '39896960',
  '51780611'
);

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5786416065374880>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "-- Query records with specific ORDERKEY values (requires casting)\nSELECT\n  variant_get(V, '$.L_ORDERKEY') AS order_key,\n  variant_get(V, '$.L_PARTKEY') AS part_key,\n  variant_get(V, '$.L_SUPPKEY') AS supp_key,\n  variant_get(V, '$.L_LINENUMBER') AS line_number,\n  variant_get(V, '$.L_QUANTITY') AS quantity,\n  variant_get(V, '$.L_EXTENDEDPRICE') AS extended_price,\n  variant_get(V, '$.L_DISCOUNT') AS discount,\n  variant_get(V, '$.L_TAX') AS tax,\n  variant_get(V, '$.L_RETURNFLAG') AS return_flag,\n  variant_get(V, '$.L_LINESTATUS') AS line_status,\n  variant_get(V, '$.L_SHIPDATE') AS ship_date,\n  variant_get(V, '$.L_COMMITDATE') AS commit_date,\n  variant_get(V, '$.L_RECEIPTDATE') AS receipt_date,\n  variant_get(V, '$.L_SHIPINSTRUCT') AS ship_

## 9. Create Structured Iceberg Table

We'll now transform the semi-structured variant data into a fully typed Iceberg table for improved query performance.

In [0]:
%sql
-- Create the final structured Iceberg table
CREATE OR REPLACE TABLE 05_CONCURRENCY_ICEBERG.default.LINEITEMS_WAREHOUSE
USING ICEBERG
AS
SELECT
  CAST(variant_get(V, '$.L_ORDERKEY') AS VARCHAR(15)) AS L_ORDERKEY,
  CAST(variant_get(V, '$.L_PARTKEY') AS VARCHAR(15)) AS L_PARTKEY,
  CAST(variant_get(V, '$.L_SUPPKEY') AS VARCHAR(15)) AS L_SUPPKEY,
  CAST(variant_get(V, '$.L_LINENUMBER') AS INT) AS L_LINENUMBER,
  CAST(variant_get(V, '$.L_QUANTITY') AS FLOAT) AS L_QUANTITY,
  CAST(variant_get(V, '$.L_EXTENDEDPRICE') AS FLOAT) AS L_EXTENDEDPRICE,
  CAST(variant_get(V, '$.L_DISCOUNT') AS FLOAT) AS L_DISCOUNT,
  CAST(variant_get(V, '$.L_TAX') AS FLOAT) AS L_TAX,
  CAST(variant_get(V, '$.L_RETURNFLAG') AS VARCHAR(30)) AS L_RETURNFLAG,
  CAST(variant_get(V, '$.L_LINESTATUS') AS VARCHAR(30)) AS L_LINESTATUS,
  CAST(variant_get(V, '$.L_SHIPDATE') AS DATE) AS L_SHIPDATE,
  CAST(variant_get(V, '$.L_COMMITDATE') AS DATE) AS L_COMMITDATE,
  CAST(variant_get(V, '$.L_RECEIPTDATE') AS DATE) AS L_RECEIPTDATE,
  CAST(variant_get(V, '$.L_SHIPINSTRUCT') AS VARCHAR(30)) AS L_SHIPINSTRUCT,
  CAST(variant_get(V, '$.L_SHIPMODE') AS VARCHAR(30)) AS L_SHIPMODE,
  CAST(variant_get(V, '$.L_COMMENT') AS VARCHAR(100)) AS L_COMMENT
FROM 05_CONCURRENCY_ICEBERG.default.raw_lineitems_warehouse;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Validate the final table
SELECT * FROM 05_CONCURRENCY_ICEBERG.default.lineitems_warehouse LIMIT 10;

L_ORDERKEY,L_PARTKEY,L_SUPPKEY,L_LINENUMBER,L_QUANTITY,L_EXTENDEDPRICE,L_DISCOUNT,L_TAX,L_RETURNFLAG,L_LINESTATUS,L_SHIPDATE,L_COMMITDATE,L_RECEIPTDATE,L_SHIPINSTRUCT,L_SHIPMODE,L_COMMENT
2840839,1859794,84813,3,4.0,7014.8,0.03,0.0,N,O,1997-05-13,1997-05-25,1997-05-30,TAKE BACK RETURN,SHIP,de of the blithely even theodoli
2840839,1553863,3894,4,33.0,63254.07,0.09,0.07,N,O,1997-06-21,1997-05-31,1997-07-18,DELIVER IN PERSON,MAIL,unusual pint
2840839,1760680,35732,5,39.0,67883.4,0.0,0.07,N,O,1997-06-09,1997-04-29,1997-07-08,TAKE BACK RETURN,MAIL,"bout the special, special accounts"
2840864,154900,79902,1,22.0,43007.8,0.0,0.04,N,O,1998-03-05,1998-01-31,1998-03-10,NONE,MAIL,ly on the car
2840864,1402556,27571,2,15.0,21877.2,0.06,0.04,N,O,1998-03-08,1998-02-24,1998-03-26,DELIVER IN PERSON,RAIL,"usly regular packages. regular, specia"
2840864,747657,72665,3,29.0,49433.98,0.08,0.05,N,O,1998-01-11,1998-03-07,1998-01-18,COLLECT COD,FOB,courts are blithely across the carefu
2840864,1867935,17972,4,42.0,79919.28,0.1,0.02,N,O,1998-04-06,1998-01-28,1998-04-22,DELIVER IN PERSON,AIR,s boost quic
2840864,381775,56785,5,19.0,35278.44,0.03,0.06,N,O,1998-02-15,1998-02-23,1998-02-24,DELIVER IN PERSON,SHIP,uickly express theodolites. carefull
2840864,492368,42377,6,44.0,59854.96,0.07,0.07,N,O,1998-02-01,1998-03-11,1998-03-01,NONE,TRUCK,n theodolites wake slyly? furiousl
2840864,1222353,22354,7,48.0,61213.92,0.06,0.05,N,O,1998-01-27,1998-02-24,1998-02-10,DELIVER IN PERSON,MAIL,ets sleep furiously. reg


## 10. Performance Testing and Validation

You can now run concurrency tests against the structured Iceberg table. Suggested queries:
- Random sampling
- Aggregations on quantity and price
- Filtering by date ranges
- Joining with dimension tables (if available)

Below are placeholders for performance-focused queries.

In [0]:
%sql
-- Count records in the final table
SELECT COUNT(*) FROM 05_CONCURRENCY_ICEBERG.default.lineitems_warehouse;

COUNT(*)
59986052


In [0]:
%sql
-- Example aggregation for performance testing
SELECT L_SHIPMODE, COUNT(*) AS cnt, SUM(L_EXTENDEDPRICE * (1 - L_DISCOUNT)) AS revenue
FROM 05_CONCURRENCY_ICEBERG.default.lineitems_warehouse
GROUP BY L_SHIPMODE
ORDER BY revenue DESC;

L_SHIPMODE,cnt,revenue
REG AIR,8570280,3.1145566248135626E11
SHIP,8571402,3.1139558495402954E11
RAIL,8571844,3.1135326535339606E11
MAIL,8569053,3.1128836024904877E11
AIR,8566164,3.112605768143045E11
FOB,8569760,3.112039311864605E11
TRUCK,8567549,3.1115732469206976E11


## 11. Cleanup Operations

Use the following commands to clean up all objects created during the benchmark.

In [0]:
%sql
-- CLEAN UP (uncomment as needed)
-- USE CATALOG 05_CONCURRENCY_ICEBERG;
-- USE SCHEMA default;

-- DROP TABLE IF EXISTS 05_CONCURRENCY_ICEBERG.default.lineitems;
-- DROP TABLE IF EXISTS 05_CONCURRENCY_ICEBERG.default.lineitems_warehouse;
-- DROP TABLE IF EXISTS 05_CONCURRENCY_ICEBERG.default.raw_lineitems;
-- DROP TABLE IF EXISTS 05_CONCURRENCY_ICEBERG.default.raw_lineitems_warehouse;
-- DROP TABLE IF EXISTS 05_CONCURRENCY_ICEBERG.default.raw_lineitems_dlt;

-- DROP SCHEMA IF EXISTS 05_CONCURRENCY_ICEBERG.default CASCADE;
-- DROP SCHEMA IF EXISTS 05_CONCURRENCY_ICEBERG.silver CASCADE;
-- DROP SCHEMA IF EXISTS 05_CONCURRENCY_ICEBERG.default CASCADE;